In [ ]:
# Cell 1: Dual Video Generation for Baseline and VLM-PPO
from pathlib import Path
import gymnasium as gym
from stable_baselines3 import PPO
import imageio.v2 as imageio
from IPython.display import Video, display
import torch

log_dir = Path("logs_and_results")
baseline_model_path = log_dir / "baseline_multi_thresh.zip"
vlm_model_path = log_dir / "vlm_multi_thresh.zip"
baseline_video_path = log_dir / "baseline_multi_thresh_demo.mp4"
vlm_video_path = log_dir / "vlm_multi_thresh_demo.mp4"

print("Generating demo videos for both models...")
video_env = gym.make("MountainCar-v0", render_mode="rgb_array")
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Baseline Video
print("Loading Baseline model...")
baseline_model = PPO.load(str(baseline_model_path), device=device)
obs, info = video_env.reset()
terminated, truncated = False, False
baseline_frames = []

while not (terminated or truncated):
    baseline_frames.append(video_env.render())
    action, _ = baseline_model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = video_env.step(action)

imageio.mimsave(baseline_video_path, baseline_frames, fps=30)
print(f"Saved Baseline video to {baseline_video_path}")

# 2. VLM-PPO Video
print("Loading VLM-PPO model...")
vlm_model = PPO.load(str(vlm_model_path), device=device)
obs, info = video_env.reset()
terminated, truncated = False, False
vlm_frames = []

while not (terminated or truncated):
    vlm_frames.append(video_env.render())
    action, _ = vlm_model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = video_env.step(action)

imageio.mimsave(vlm_video_path, vlm_frames, fps=30)
print(f"Saved VLM-PPO video to {vlm_video_path}")

video_env.close()

# 3. Display Videos
print("\n--- Baseline Model Demo ---")
display(Video(str(baseline_video_path), embed=True))

print("\n--- VLM-PPO Model Demo ---")
display(Video(str(vlm_video_path), embed=True))


In [ ]:
# Cell 2: JSON-based Comparison Chart
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

thresholds = [-195, -190, -180, -170, -160]
baseline_dnf_cap = 600000
vlm_dnf_cap = 150000

log_dir = Path("logs_and_results")
baseline_json = log_dir / "baseline_thresholds.json"
vlm_json = log_dir / "vlm_thresholds.json"

if not baseline_json.exists() or not vlm_json.exists():
    raise FileNotFoundError(
        f"Missing JSON files. Expected {baseline_json} and {vlm_json}."
    )

with open(baseline_json, "r", encoding="utf-8") as f:
    baseline_data = json.load(f)
with open(vlm_json, "r", encoding="utf-8") as f:
    vlm_data = json.load(f)

# Combine missing/null defaults
rows = []
for t in thresholds:
    key = str(t)
    baseline_steps = baseline_data.get(key)
    if baseline_steps is None:
        baseline_steps = baseline_dnf_cap
    else:
        baseline_steps = int(baseline_steps)
        
    vlm_steps = vlm_data.get(key)
    if vlm_steps is None:
        vlm_steps = vlm_dnf_cap
    else:
        vlm_steps = int(vlm_steps)

    speedup = baseline_steps / vlm_steps if vlm_steps > 0 else np.nan
    rows.append(
        {
            "Threshold": t,
            "Baseline_Steps": baseline_steps,
            "VLM_Steps": vlm_steps,
            "Speedup": speedup,
        }
    )

comparison_df = pd.DataFrame(rows)
display(comparison_df)

x = np.arange(len(thresholds))
bar_width = 0.35

fig, ax = plt.subplots(figsize=(11, 6))
bars_baseline = ax.bar(
    x - bar_width / 2,
    comparison_df["Baseline_Steps"],
    width=bar_width,
    label="Baseline",
    color="#4C78A8",
)
bars_vlm = ax.bar(
    x + bar_width / 2,
    comparison_df["VLM_Steps"],
    width=bar_width,
    label="VLM-PPO",
    color="#E45756",
)

# Annotate DNF for Baseline
for i, b in enumerate(bars_baseline):
    val = int(comparison_df.loc[i, "Baseline_Steps"])
    if val >= baseline_dnf_cap:
        ax.text(
            b.get_x() + b.get_width() / 2,
            b.get_height() + 5000,
            "DNF (>600k)",
            ha="center",
            va="bottom",
            color="red",
            fontsize=10,
            fontweight="bold",
        )

# Annotate DNF for VLM-PPO
for i, b in enumerate(bars_vlm):
    val = int(comparison_df.loc[i, "VLM_Steps"])
    if val >= vlm_dnf_cap:
        ax.text(
            b.get_x() + b.get_width() / 2,
            b.get_height() + 5000,
            "DNF (>150k)",
            ha="center",
            va="bottom",
            color="red",
            fontsize=10,
            fontweight="bold",
        )

ax.set_xticks(x)
ax.set_xticklabels([str(t) for t in thresholds])
ax.set_xlabel("Reward Threshold")
ax.set_ylabel("Steps to Reach Threshold")
ax.set_title("Performance Comparison: Baseline vs VLM-PPO")
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.7)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()
